In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df_sample = pd.read_csv("/Volumes/workspace/default/echantillon/yellowtaxisample1pct_hybrid_stratified.csv")

df_sample.info()
df_sample.head()




In [0]:
#Conversion des dates
df_sample["tpep_pickup_datetime"] = pd.to_datetime(
    df_sample["tpep_pickup_datetime"], errors="coerce"
)

df_sample["tpep_dropoff_datetime"] = pd.to_datetime(
    df_sample["tpep_dropoff_datetime"], errors="coerce"
)
df_sample[["tpep_pickup_datetime", "tpep_dropoff_datetime"]].head()


In [0]:
# Détection des dates invalides (pickup > dropoff)
invalid_dates = df_sample[
    df_sample["tpep_dropoff_datetime"] < df_sample["tpep_pickup_datetime"]
]

print("Nombre de lignes avec dates invalides :", len(invalid_dates))

invalid_dates.head()

In [0]:
#Correction des dates inversées
mask = df_sample["tpep_pickup_datetime"] > df_sample["tpep_dropoff_datetime"]

df_sample.loc[mask, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]] = (
    df_sample.loc[mask, ["tpep_dropoff_datetime", "tpep_pickup_datetime"]].values
)

In [0]:
#Suppression des doublons
df_sample = df_sample.drop_duplicates()
df_sample.shape

In [0]:
#Nettoyage airport_fee
if "Airport_fee" in df_sample.columns:
    df_sample["airport_fee"] = df_sample["airport_fee"].fillna(
        df_sample["Airport_fee"]
    )
    df_sample.drop(["Airport_fee"], axis=1, inplace=True)

df_sample["airport_fee"] = df_sample["airport_fee"].fillna(0.0)

In [0]:
#Nettoyage passenger_count
df_sample["passenger_count"] = df_sample["passenger_count"].fillna(1)


In [0]:
#Nettoyage RatecodeID
df_sample["RatecodeID"] = df_sample["RatecodeID"].replace(99, pd.NA)
df_sample["RatecodeID"] = df_sample["RatecodeID"].fillna(-1).astype(int)


In [0]:
#Nettoyage store_and_fwd_flag
df_sample["store_and_fwd_flag"] = df_sample["store_and_fwd_flag"].fillna("N")


In [0]:
#Nettoyage congestion_surcharge
df_sample["congestion_surcharge"] = df_sample["congestion_surcharge"].fillna(0.0)


In [0]:
df_sample.info()
df_sample.isna().sum()
df_sample.head()


# ## **STATISTIQUES**

Prix moyen d’une course

In [0]:
#Comparer l’estimation du sample vs la valeur exacte sur population
mean_fare_sample = df_sample["fare_amount"].mean()
mean_fare_sample

In [0]:
n = df_sample.shape[0]
mean_fare = df_sample["fare_amount"].mean()
std_fare = df_sample["fare_amount"].std()

z = 1.96
margin_fare = z * (std_fare / np.sqrt(n))

ic_fare = (mean_fare - margin_fare, mean_fare + margin_fare)

print(f"Prix moyen : {mean_fare:.2f}")
print(f"IC 95 % : [{ic_fare[0]:.2f} ; {ic_fare[1]:.2f}]")

Distance moyenne d’une course

In [0]:
#L’échantillon est-il représentatif des distances réelles ?
mean_distance_sample = df_sample["trip_distance"].mean()
mean_distance_sample


In [0]:
mean_dist = df_sample["trip_distance"].mean()
std_dist = df_sample["trip_distance"].std()

margin_dist = z * (std_dist / np.sqrt(n))
ic_dist = (mean_dist - margin_dist, mean_dist + margin_dist)

print(f"Distance moyenne : {mean_dist:.2f}")
print(f"IC 95 % : [{ic_dist[0]:.2f} ; {ic_dist[1]:.2f}]")


Durée moyenne des courses

In [0]:
#Peut-odf_sample["trip_duration_min"] = (
df_sample["trip_duration_min"] = (
    (df_sample["tpep_dropoff_datetime"] - df_sample["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)

mean_duration_sample = df_sample["trip_duration_min"].mean()
mean_duration_sample


In [0]:
df_sample["trip_duration_min"] = (
    (df_sample["tpep_dropoff_datetime"] - df_sample["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)

mean_dur = df_sample["trip_duration_min"].mean()
std_dur = df_sample["trip_duration_min"].std()

margin_dur = z * (std_dur / np.sqrt(n))
ic_dur = (mean_dur - margin_dur, mean_dur + margin_dur)

print(f"Durée moyenne (min) : {mean_dur:.2f}")
print(f"IC 95 % : [{ic_dur[0]:.2f} ; {ic_dur[1]:.2f}]")


Proportion des courses avec tip > 0

In [0]:
#Inférence vs valeur réelle? 
df_sample["has_tip"] = (df_sample["tip_amount"] > 0).astype(int)
tip_rate_sample = df_sample["has_tip"].mean()
tip_rate_sample * 100


In [0]:
df_sample["has_tip"] = (df_sample["tip_amount"] > 0).astype(int)

p = df_sample["has_tip"].mean()

margin_tip = z * np.sqrt((p * (1 - p)) / n)
ic_tip = (p - margin_tip, p + margin_tip)

print(f"Proportion tip > 0 : {p*100:.2f} %")
print(f"IC 95 % : [{ic_tip[0]*100:.2f} % ; {ic_tip[1]*100:.2f} %]")


Distribution par heure / jour / semaine

In [0]:
#Identifier les heures de pointe
#Nombre de courses par heure
# 1️⃣ Ajouter les colonnes nécessaires
df_sample["pickup_hour"] = df_sample["tpep_pickup_datetime"].dt.hour
df_sample["pickup_day_name"] = df_sample["tpep_pickup_datetime"].dt.day_name()
df_sample["pickup_week"] = df_sample["tpep_pickup_datetime"].dt.isocalendar().week

# 2️⃣ Nombre de courses par heure
rides_hour = df_sample["pickup_hour"].value_counts().sort_index()
print("📌 Nombre de courses par heure :")
for hour, count in rides_hour.items():
    print(f"Heure {hour}: {count} courses")

# 3️⃣ Nombre de courses par jour
rides_day = df_sample["pickup_day_name"].value_counts().reindex([
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
])
print("\n📌 Nombre de courses par jour :")
for day, count in rides_day.items():
    print(f"{day}: {count} courses")

# 4️⃣ Nombre de courses par semaine
rides_week = df_sample["pickup_week"].value_counts().sort_index()
print("\n📌 Nombre de courses par semaine :")
for week, count in rides_week.items():
    print(f"Semaine {week}: {count} courses")

Comparaison géographique (zones / boroughs)


In [0]:
#L’échantillon reflète-t-il la diversité spatiale ?

# 2️⃣ Vérifier que les colonnes existent
print(df_sample.columns)  # assure-toi qu'il y a bien 'PULocationID' et 'fare_amount'

# 3️⃣ Calculer le prix moyen par zone de départ
prix_par_zone_depart = (
    df_sample.groupby("PULocationID")["fare_amount"]
    .mean()
    .sort_values(ascending=False)
)

# 4️⃣ Afficher le résultat
print(prix_par_zone_depart.head(10)) 

Analyse des outliers

In [0]:
#Impact sur estimation vs population
q1 = df_sample["fare_amount"].quantile(0.25)
q3 = df_sample["fare_amount"].quantile(0.75)
iqr = q3 - q1

outliers_sample = df_sample[
    (df_sample["fare_amount"] < q1 - 1.5 * iqr) |
    (df_sample["fare_amount"] > q3 + 1.5 * iqr)
]

len(outliers_sample)

Ratio tip / fare par type de paiement

In [0]:
#Cash vs Card
df_sample["tip_ratio"] = df_sample["tip_amount"] / df_sample["fare_amount"]

df_sample.groupby("payment_type")["tip_ratio"].mean()


In [0]:
fare_by_PU_sample = df_sample.groupby("PULocationID").agg(
    avg_fare=("fare_amount", "mean"),
    nb_trips=("fare_amount", "count")
).reset_index()

In [0]:
fare_by_PU_sample.head(10).plot(
    kind="bar",
    x="PULocationID",
    y="avg_fare",
    color="pink",
    figsize=(10,5),
    legend=False
)

plt.title("Prix moyen par zone de pickup – Échantillon (1%)")
plt.xlabel("Pickup LocationID")
plt.ylabel("Prix moyen ($)")
plt.tight_layout()
plt.show()